In [1]:
import pandas as pd

In [2]:
file_path = 'All_external.csv'
output_path = 'filtered_tech_news_2018_2020.csv'
target_tickers = ['NVDA', 'AAPL', 'MSF', 'GOOGL', 'FB', 'AMZN', 'TSLA']
chunk_size = 100000

filtered_chunks = []

In [3]:
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)):
    if 'Stock_symbol' in chunk.columns:
        ticker_col = 'Stock_symbol'
    else:
        ticker_col = [col for col in chunk.columns if 'stock' in col.lower() or 'symbol' in col.lower() or 'ticker' in col.lower()][0]

    filtered_chunk = chunk[chunk[ticker_col].isin(target_tickers)].copy()
    
    if not filtered_chunk.empty:
        date_col = [col for col in filtered_chunk.columns if 'date' in col.lower()][0]
        
        filtered_chunk['Date_Clean'] = pd.to_datetime(filtered_chunk[date_col], errors='coerce')
        filtered_chunk = filtered_chunk[
            (filtered_chunk['Date_Clean'] >= '2018-01-01') & 
            (filtered_chunk['Date_Clean'] <= '2020-12-31')
        ]
        filtered_chunk[ticker_col] = filtered_chunk[ticker_col].replace({'FB': 'META', 'MSF': 'MSFT'})
        
        filtered_chunks.append(filtered_chunk)

    if (i+1) % 10 == 0:
        print(f"Processed {i * chunk_size} rows...")

Processed 900000 rows...
Processed 1900000 rows...
Processed 2900000 rows...
Processed 3900000 rows...
Processed 4900000 rows...
Processed 5900000 rows...
Processed 6900000 rows...
Processed 7900000 rows...
Processed 8900000 rows...
Processed 9900000 rows...
Processed 10900000 rows...
Processed 11900000 rows...
Processed 12900000 rows...


In [4]:
df_final = pd.concat(filtered_chunks)
df_final.to_csv(output_path, index=False)